# 한국어 우회 공격 탐지 실험 - 소형 모델 3종

이 노트북은 Kaggle GPU 환경에서 KLUE-BERT, KLUE-RoBERTa, KCBERT를 학습하고 원본 test와 변형 test 성능을 비교하기 위한 실행 노트북입니다.

실행 순서:
1. 프로젝트 경로와 GPU 확인
2. 필요한 패키지 설치
3. 데이터 파일 확인
4. 변형 데이터가 없으면 공격 데이터 생성
5. 소형 모델 3개 학습 및 평가
6. 결과 시각화
7. 결과 파일 압축

In [ ]:
from pathlib import Path
import os

# Kaggle에 업로드한 프로젝트 폴더 위치에 맞게 필요하면 수정하세요.
PROJECT_DIR = Path('/kaggle/working/korean-adversarial-nlp')

if not PROJECT_DIR.exists():
    # zip을 풀지 않고 파일이 바로 /kaggle/working에 있는 경우를 대비합니다.
    PROJECT_DIR = Path('/kaggle/working')

os.chdir(PROJECT_DIR)
print('프로젝트 경로:', PROJECT_DIR)
print('현재 파일 목록:')
!ls -la

In [ ]:
# GPU가 켜져 있는지 확인합니다.
!nvidia-smi

In [ ]:
# Kaggle 기본 환경과 충돌을 줄이기 위해 핵심 패키지만 확인/설치합니다.
# raw 데이터부터 재생성할 경우 requirements.txt 전체 설치가 필요할 수 있습니다.
!pip install -q transformers==4.40.0 datasets==2.19.0 pandas==2.2.2 numpy==1.26.4 scikit-learn==1.4.2 matplotlib==3.8.4 seaborn==0.13.2 tqdm==4.66.2

In [ ]:
# 학습에 필요한 processed 데이터와 평가에 필요한 augmented 데이터를 확인합니다.
from pathlib import Path

required_files = [
    Path('data/processed/train.csv'),
    Path('data/processed/val.csv'),
    Path('data/processed/test.csv'),
]

for file_path in required_files:
    print(file_path, '존재' if file_path.exists() else '없음')

augmented_files = sorted(Path('data/augmented').glob('test_*.csv'))
print('변형 test 파일 수:', len(augmented_files))
print('예시:', [p.name for p in augmented_files[:5]])

In [ ]:
# processed 데이터가 없다면 raw 데이터에서 전처리를 수행합니다.
# 이미 data/processed/*.csv를 업로드했다면 이 셀은 자동으로 건너뜁니다.
from pathlib import Path

processed_ready = all(Path(p).exists() for p in [
    'data/processed/train.csv',
    'data/processed/val.csv',
    'data/processed/test.csv',
])

if processed_ready:
    print('processed 데이터가 이미 있어 전처리를 건너뜁니다.')
else:
    print('processed 데이터가 없어 전처리를 실행합니다.')
    !python src/utils/preprocess.py

In [ ]:
# 변형 test 데이터가 없다면 단일 공격 9종 x 강도 3단계 데이터를 생성합니다.
# 이미 data/augmented/test_*.csv를 업로드했다면 이 셀은 자동으로 건너뜁니다.
from pathlib import Path

augmented_files = sorted(Path('data/augmented').glob('test_*.csv'))
if augmented_files:
    print(f'변형 test 파일 {len(augmented_files)}개가 이미 있어 공격 생성을 건너뜁니다.')
else:
    print('변형 test 파일이 없어 공격 데이터를 생성합니다.')
    !python src/attacks/run_all_attacks.py

In [ ]:
# 소형 모델 3개를 순서대로 학습하고 원본 test/변형 test 평가 결과를 저장합니다.
# 기본값은 variant_id=1만 평가합니다.
# 모든 variant를 평가하려면 뒤에 --all_variants를 붙이세요.
# 결과: results/metrics/klue-bert_results.csv 등
!python src/models/small_model.py --model all

In [ ]:
# 저장된 metrics CSV를 확인합니다.
import pandas as pd
from pathlib import Path

metric_files = sorted(Path('results/metrics').glob('*_results.csv'))
print('결과 파일:', [p.name for p in metric_files])

if metric_files:
    preview = pd.concat([pd.read_csv(p).head(3) for p in metric_files], ignore_index=True)
    display(preview)

In [ ]:
# 결과 그래프를 생성합니다.
# 결과: results/figures/*.png
!python src/evaluation/visualize.py

In [ ]:
# 생성된 그래프 파일을 확인합니다.
from pathlib import Path

figure_files = sorted(Path('results/figures').glob('*.png'))
print('그래프 파일:', [p.name for p in figure_files])

In [ ]:
# Kaggle Output에서 내려받기 쉽도록 결과 폴더를 압축합니다.
!zip -r results_small_models.zip results/metrics results/figures